# 3-3. Advanced RAG: Evidence Selection

## 학습 목표

- 검색된 문서 각각을 LLM으로 평가하여 질문과 관련 있는 근거만 선별한다.
- 관련 없는 문서를 답변 생성 전에 필터링하여 노이즈를 제거한다.
- 각 문서에서 질문에 직접 관련된 증거 문장만 추출한다.

## 공통 전제

- 실습 데이터: `../data/공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md`
- 기본 모델: `gpt-4o-mini`
- 기본 임베딩 모델: `text-embedding-3-small`
- 벡터DB: Qdrant Docker Server (`http://localhost:6333`)

> 이 노트북은 `2-2_vector_embedding_store_qdrant_md.ipynb`에서 저장한 Qdrant collection을 재사용한다.

## Evidence Selection 이란?

| 전략 | 설명 |
|------|------|
| Reranking | 검색 결과의 **순서**를 재조정 |
| Context Compression | 문서 내용을 **압축**하여 핵심만 요약 |
| **Evidence Selection** | 문서 단위로 **관련 여부를 판단**하고, 관련 없는 문서를 **제거** |

### 처리 흐름

```
검색 (k개 문서)
    ↓
문서별 Relevance Scoring (LLM 평가)
    ↓
임계값(threshold) 이상만 필터링
    ↓
관련 증거 문장 추출 (Evidence Extraction)
    ↓
답변 생성
```

### Reranking vs Evidence Selection

- **Reranking**: 모든 문서를 순서만 바꿔서 LLM에 전달 → 여전히 노이즈 문서 포함
- **Evidence Selection**: 관련 없는 문서를 **완전히 제거** → LLM이 더 깔끔한 컨텍스트를 받음

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../../.env")

## 1. 검색 결과 가져오기

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "civil_complaint_manual_medium"
EMBEDDING_MODEL = "text-embedding-3-small"

# 질문을 벡터로 변환할 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# 이미 저장된 Qdrant 컬렉션에 연결 (새로 생성하지 않음)
vector_store = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    url=QDRANT_URL,
)

In [ ]:
question = "권장시간이 지났는데도 계속 상담을 요구하면 어떻게 하나요?"

# k=6: 상위 6개 문서를 코사인 유사도 기준으로 검색
# Evidence Selection은 이후 단계에서 노이즈를 걸러내므로, k를 넉넉히 설정하는 것이 일반적
docs = vector_store.similarity_search(question, k=6)

for i, doc in enumerate(docs, start=1):
    print(f"--- 검색 결과 {i} ---")
    print(f"page={doc.metadata.get('page')}, topic={doc.metadata.get('topic')}")
    print(doc.page_content[:300].replace("\n", " "))
    print()

## 2. Evidence Scoring: 문서별 관련도 평가

각 문서가 질문에 답하는 데 **얼마나 관련 있는지** LLM이 0~10점으로 평가한다.

- **점수 기준**: 0 = 전혀 무관, 10 = 직접적인 근거 포함
- **이유(reason)**: 점수 판단 근거를 함께 반환 → 설명 가능성(explainability) 확보

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.documents import Document
from pydantic import BaseModel, Field

# Pydantic 스키마: LLM 출력을 구조화된 JSON으로 파싱하기 위한 데이터 모델
# JsonOutputParser와 함께 사용하여 타입 안전성 확보
class EvidenceScore(BaseModel):
    score: int = Field(description="질문과의 관련도 점수 (0~10)")
    reason: str = Field(description="점수 판단 이유 (1~2 문장)")
    is_relevant: bool = Field(description="관련 있으면 True (score >= 5)")

# 문서별 관련도 평가 프롬프트
# system: LLM에게 Evidence Selector 역할 부여 및 점수 기준 명시
# human: 평가할 질문과 문서를 동적으로 주입
scoring_prompt = ChatPromptTemplate.from_messages([
    ("system", """
당신은 RAG 시스템의 Evidence Selector입니다.
주어진 문서가 질문에 답하는 데 얼마나 관련 있는지 평가하세요.

점수 기준:
- 9~10: 질문에 대한 직접적인 답변 근거 포함
- 7~8: 관련 있지만 일부 간접적인 내용
- 5~6: 부분적으로 관련 있음
- 3~4: 약한 연관성
- 0~2: 질문과 무관

반드시 JSON 형식으로 응답하세요:
{{"score": <0~10>, "reason": "<판단 이유>", "is_relevant": <true/false>}}
"""),
    ("human", """
질문: {question}

문서 내용:
{document}
""")
])

# temperature=0: 점수 평가는 일관성이 중요하므로 무작위성 제거
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 체인 구성: 프롬프트 → LLM → JSON 파싱
# JsonOutputParser가 LLM 출력 문자열을 Python dict로 자동 변환
scoring_chain = scoring_prompt | llm | JsonOutputParser()

print("Evidence Scoring 준비 완료")

In [ ]:
scored_docs = []

# 검색된 문서 각각에 대해 LLM 관련도 평가 (k번 LLM 호출 발생)
for i, doc in enumerate(docs, start=1):
    result = scoring_chain.invoke({
        "question": question,
        "document": doc.page_content
    })
    
    # 원본 Document 객체와 LLM 평가 결과를 함께 저장
    scored_docs.append({
        "doc": doc,
        "score": result["score"],
        "reason": result["reason"],
        "is_relevant": result["is_relevant"]
    })
    
    relevance_label = "✅ 관련 있음" if result["is_relevant"] else "❌ 관련 없음"
    print(f"--- 문서 {i} | 점수: {result['score']}/10 | {relevance_label} ---")
    print(f"  page={doc.metadata.get('page')}, topic={doc.metadata.get('topic')}")
    print(f"  판단 이유: {result['reason']}")
    print()

## 3. Evidence Filtering: 임계값 기반 필터링

점수가 임계값(threshold) **미만인 문서는 제거**한다.

- 임계값이 높을수록 → 더 엄격한 필터링 (정밀도↑, 재현율↓)
- 임계값이 낮을수록 → 더 느슨한 필터링 (재현율↑, 노이즈↑)

In [ ]:
THRESHOLD = 5  # 5점 미만은 제거 (0~10점 척도에서 중간값을 기준으로 설정)

# 고득점 문서가 먼저 오도록 내림차순 정렬 → 답변 생성 시 우선순위 반영
scored_docs.sort(key=lambda x: x["score"], reverse=True)

# threshold 이상: 관련 있는 증거 문서 / 미만: 노이즈로 간주하여 제거
relevant_docs = [item for item in scored_docs if item["score"] >= THRESHOLD]
filtered_out  = [item for item in scored_docs if item["score"] < THRESHOLD]

print(f"전체 검색 문서: {len(docs)}개")
print(f"선별된 증거 문서: {len(relevant_docs)}개 (threshold={THRESHOLD})")
print(f"제거된 문서: {len(filtered_out)}개")
print()

if filtered_out:
    print("=== 제거된 문서 ===")
    for item in filtered_out:
        print(f"  - 점수 {item['score']}/10 | page={item['doc'].metadata.get('page')}, topic={item['doc'].metadata.get('topic')}")
    print()

print("=== 선별된 증거 문서 ===")
for i, item in enumerate(relevant_docs, start=1):
    print(f"  {i}. 점수 {item['score']}/10 | page={item['doc'].metadata.get('page')}, topic={item['doc'].metadata.get('topic')}")

## 4. Evidence Extraction: 관련 증거 문장 추출

선별된 문서에서도 **질문과 직접 관련된 문장만** 추출한다.

- 문서 전체를 그대로 쓰지 않고, 핵심 증거 문장만 뽑는다.
- Context Compression과의 차이: 요약이 아닌 **원문 발췌**에 가까움

In [ ]:
extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", """
당신은 RAG 시스템의 Evidence Extractor입니다.
주어진 문서에서 질문에 직접 관련된 문장이나 구절만 추출하세요.

규칙:
- 원문에 있는 내용만 추출하세요. 재작성하거나 요약하지 마세요.
- 질문과 무관한 문장은 포함하지 마세요.
- 추출한 내용을 bullet 형식으로 나열하세요.
- 관련 내용이 없으면 "관련 없음"이라고 답하세요.
"""),
    ("human", """
질문: {question}

문서 내용:
{document}

위 문서에서 질문에 관련된 문장만 추출하세요.
""")
])

from langchain_core.output_parsers import StrOutputParser

# StrOutputParser: 추출 결과는 자유 형식 텍스트이므로 JSON 파싱 없이 문자열로 수신
extraction_chain = extraction_prompt | llm | StrOutputParser()

print("Evidence Extraction 준비 완료")

In [ ]:
evidences = []

# 필터링된 관련 문서(relevant_docs)에 대해서만 증거 추출 수행
# → 이미 노이즈 문서가 제거된 상태이므로 추출 품질이 높음
for i, item in enumerate(relevant_docs, start=1):
    extracted = extraction_chain.invoke({
        "question": question,
        "document": item["doc"].page_content
    })
    
    # 원본 문서, 관련도 점수, 추출된 증거 문장을 묶어 저장
    evidences.append({
        "doc": item["doc"],
        "score": item["score"],
        "extracted": extracted
    })
    
    print(f"=== 증거 {i} (점수: {item['score']}/10, page={item['doc'].metadata.get('page')}) ===")
    print(extracted)
    print()

## 5. 선별된 증거로 답변 생성

필터링과 추출을 거친 **고품질 증거**만으로 최종 답변을 생성한다.

In [ ]:
def format_evidences(evidences: list[dict]) -> str:
    """추출된 증거들을 LLM 컨텍스트로 조합.
    
    각 증거에 출처(page, topic)와 관련도 점수를 명시하여
    LLM이 근거 기반 답변을 생성할 수 있도록 한다.
    """
    parts = []
    for i, ev in enumerate(evidences, start=1):
        meta = ev["doc"].metadata
        parts.append(
            f"[증거 {i}] page={meta.get('page')}, topic={meta.get('topic')}, "
            f"관련도={ev['score']}/10\n{ev['extracted']}"
        )
    # 증거들을 빈 줄로 구분하여 가독성 확보
    return "\n\n".join(parts)

evidence_context = format_evidences(evidences)
print(evidence_context)

In [ ]:
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", """
당신은 공직자 민원응대 매뉴얼 기반 업무지원 AI입니다.
제공된 근거를 바탕으로만 답변하세요. 질문에 대한 근거가 없으면 없다고 대답하세요.

답변 형식:
1. 핵심 대응
2. 단계별 조치
3. 안내 표현
4. 주의사항
"""),
    ("human", """
질문:
{question}

선별된 근거:
{context}
""")
])

answer_chain = answer_prompt | llm | StrOutputParser()

# evidence_context: Scoring → Filtering → Extraction을 거친 고품질 컨텍스트
# 노이즈 문서가 제거된 상태이므로 hallucination 위험이 낮음
answer = answer_chain.invoke({
    "question": question,
    "context": evidence_context
})

print(answer)

## 6. 전체 파이프라인 통합

Evidence Selection의 세 단계를 하나의 함수로 묶는다.

In [ ]:
def evidence_selection_rag(
    question: str,
    k: int = 6,
    threshold: int = 5
) -> str:
    """
    Evidence Selection RAG 파이프라인
    1. Retrieve: 벡터 검색으로 k개 문서 수집
    2. Score: 각 문서의 관련도를 LLM으로 평가
    3. Filter: threshold 미만 문서 제거
    4. Extract: 관련 증거 문장 추출
    5. Generate: 정제된 증거로 답변 생성
    """
    # Step 1. Retrieve: 벡터 유사도 검색
    docs = vector_store.similarity_search(question, k=k)
    print(f"[1] Retrieve: {len(docs)}개 문서 검색")

    # Step 2. Score: 문서마다 LLM 호출로 관련도 점수 산출 후 내림차순 정렬
    scored = []
    for doc in docs:
        result = scoring_chain.invoke({"question": question, "document": doc.page_content})
        scored.append({"doc": doc, "score": result["score"], "reason": result["reason"]})
    scored.sort(key=lambda x: x["score"], reverse=True)
    print(f"[2] Score 완료 | 점수 분포: {[s['score'] for s in scored]}")

    # Step 3. Filter: threshold 미만 문서 제거 → 노이즈 차단
    relevant = [item for item in scored if item["score"] >= threshold]
    print(f"[3] Filter: {len(relevant)}개 문서 선별 (threshold={threshold})")
    
    # 관련 문서가 하나도 없으면 조기 종료
    if not relevant:
        return "관련된 근거 문서를 찾지 못했습니다."

    # Step 4. Extract: 관련 문서에서 질문과 직접 관련된 문장만 발췌
    evidences = []
    for item in relevant:
        extracted = extraction_chain.invoke({"question": question, "document": item["doc"].page_content})
        evidences.append({"doc": item["doc"], "score": item["score"], "extracted": extracted})
    print(f"[4] Extract 완료")

    # Step 5. Generate: 정제된 증거 컨텍스트로 최종 답변 생성
    context = format_evidences(evidences)
    answer = answer_chain.invoke({"question": question, "context": context})
    print(f"[5] Generate 완료")

    return answer


result = evidence_selection_rag(question)
print()
print("=" * 60)
print("최종 답변")
print("=" * 60)
print(result)

## 7. 다른 질문으로 테스트

In [ ]:
question2 = "민원인이 폭언을 할 때 담당자는 어떻게 해야 하나요?"

result2 = evidence_selection_rag(question2)
print()
print("=" * 60)
print("최종 답변")
print("=" * 60)
print(result2)

## 핵심 정리

| 단계 | 작업 | 핵심 포인트 |
|------|------|-------------|
| **Scoring** | LLM이 각 문서의 관련도를 0~10점으로 평가 | 점수 + 이유 반환으로 설명 가능성 확보 |
| **Filtering** | 임계값 미만 문서 제거 | 노이즈 제거 → LLM 컨텍스트 품질 향상 |
| **Extraction** | 관련 증거 문장만 발췌 | 요약이 아닌 원문 기반 추출 |
| **Generation** | 정제된 증거로 답변 생성 | 근거 없는 환각(hallucination) 억제 |

### Evidence Selection의 장단점

**장점**
- 관련 없는 문서 제거 → LLM이 핵심 근거에 집중
- 각 문서 평가에 이유를 남겨 **설명 가능한(explainable) RAG** 구현
- 불필요한 컨텍스트 줄여 토큰 비용 절약

**단점**
- 문서별 LLM 호출 → Latency 증가 (k개 문서 = k번 LLM 호출)
- 임계값 설정에 따라 중요한 문서가 누락될 수 있음
- 단독 사용보다는 Reranking 또는 Compression과 **조합**하여 사용하는 경우가 많음